## Código para checar se a área iluminada fixa funciona.

In [ ]:
import os
import time
import sys
import clr
import numpy as np
import matplotlib.pyplot as plt
from structured_optics import *
from structured_optics.slmdisplay import My_SLM
from structured_optics.hologram import slm_hologram
import time 
from pylablib.devices import Thorlabs

from camera import *
from tqdm import tqdm


In [ ]:
slm = My_SLM(1)
cam = init_camera()

In [ ]:
#set_exposure(cam, 0.01)

In [ ]:

hg_modes = [[0,0], [0,1], [1,0], [2,0], [1,1], [0,2], [3,0], [2,1], [1,2], [0,3], [4,0], [3,1], [2,2], [1,3], [0,4], [5,0], [4,1], [3,2], [2,3], [1,4], [0,5]]
lg_modes = [[0,0], [0,1], [0,-1], [0,2], [1,0], [0,-2], [0,3], [1,1], [1,-1], [0,-3], [0,4], [1, 2], [1,-2], [2, 0], [0,-4], [0,5], [1,3], [1, -3], [2, 1], [2,-1], [0,-5]]


In [ ]:
width = slm.width
height = slm.height


pixel_size = 8e-6
beam = Beam(nix = pixel_size*width/2, Dx = width,
            niy = pixel_size*height/2, Dy = height,
            waist = 0.5e-3, lamb = 633e-9)

## Check Laguerre

In [ ]:
std_lg = []
mu = 1
for mode in range(len(lg_modes)):

    p = lg_modes[mode][0]   
    l = lg_modes[mode][1]

    beam.waist = np.sqrt(2*C**2/(2*p + np.abs(l) +1) + 2*np.sqrt(np.abs(C**4/(2*p + np.abs(l) +1)**2 - z**2/beam.k()**2)))/mu
    beam.lg(l=l, p=p)
    holo = slm_hologram(beam, 3,4)
    slm.update_image(holo)
    time.sleep(0.2)
    img = take_picture(cam)

    std = beam_std(img)['sigma_r']
    std_lg.appentaked(std)

In [ ]:
plt.plot(np.array(std_lg))
plt.ylim(0,1e-3)

## Check Hermite

In [ ]:
std_hg = []

for n in range(5):

    beam.waist = np.sqrt(2*C**2/(2*n+1) + 2*np.sqrt(C**4/(2*n+1)**2 - z**2/beam.k()**2))/mu
    print(beam.waist)
    beam.hg(n=n, m=0)
    holo = slm_hologram(beam, 3,4)
    slm.update_image(holo)
    time.sleep(0.2)

    std = beam_std(img)['sigma_x']
    std_hg.append(std)


In [ ]:
plt.plot(np.array(std_hg))
plt.ylim(0, 1e-3)

In [ ]:
slm.close()
close_camera(cam)